### GAT Grid Search as a batch ###

I want to understand more about when and why the GAT works poorly or well. To accomplish this, I'll run a grid search on meta parameters for the GAT, push the resulting embedding through a classifier, and evaluate the relationship between the meta parameters and the F1 score.

In [1]:
LOWER_CONFIDENCE = 5
EMBEDDING_DIMENSION = 8
DROP_OUT = 0.2
MAX_DEPTH = 5
NUM_EPOCHS = 800
HEADS = 8
F1_WEIGHTING_METHOD = 'Weighted' #"Macro"
VISUALIZE_EMBEDDING = False
LEARNING_RATE = 0.001

Read graph

In [2]:
import pickle
# filename = './pickle_data/tensor_type nominal NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 2.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
# filename = './pickle_data/tensor_type nominal NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 3.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
# filename = './pickle_data/tensor_type nominal NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.75 QUORUM_THRESHOLD 2.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
# filename = './pickle_data/tensor_type nominal NUM_AGENTS 20 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 4.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
# filename = './pickle_data/tensor_type time NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 2.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
# filename = './pickle_data/tensor_type time NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 3.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
filename = './pickle_data/tensor_type time NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.75 QUORUM_THRESHOLD 2.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
# filename = './pickle_data/tensor_type time NUM_AGENTS 20 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 4.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'

with open(filename, 'rb') as f:
    data = pickle.load(f)
print(data)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch_geometric/typing.py:110: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: dlopen(/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch_sparse/_version_cpu.so, 0x0006): Symbol not found: __ZN3c1017RegisterOperatorsD1Ev
  Referenced from: <594D1ECB-C389-3CE8-8CA8-529E73641ADB> /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch_sparse/_version_cpu.so
  Expected in:     <2A8DB508-8AAF-3FF1-BDFE-9EF17CC2B482> /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/lib/libtorch_cpu.dylib
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "


Data(x=[18696, 100], edge_index=[2, 84944], num_nodes=18696, y=[18696], num_classes=4, train_mask=[18696], test_mask=[18696])


Import all GAT classes.

In [3]:
from utilities.GAT_Classes import GAT_4_layer
from utilities.GAT_Classes import GAT_3_layer
from utilities.GAT_Classes import GAT_2_layer
from utilities.GAT_Classes import GAT_1_layer
def get_gats(embedding_dimension, drop_out):
    gat_4_layer = GAT_4_layer(dim_in = data.num_features, 
                            dim_h = embedding_dimension, 
                            dim_out = data.num_classes, 
                            heads = 8, 
                            drop_out=drop_out, 
                            learning_rate=LEARNING_RATE, 
                            weighting_method=F1_WEIGHTING_METHOD)
    gat_3_layer = GAT_3_layer(dim_in = data.num_features, 
                            dim_h = embedding_dimension, 
                            dim_out = data.num_classes, 
                            heads = 8, 
                            drop_out=drop_out, 
                            learning_rate=LEARNING_RATE, 
                            weighting_method=F1_WEIGHTING_METHOD)
    gat_2_layer = GAT_2_layer(dim_in = data.num_features, 
                            dim_h = embedding_dimension, 
                            dim_out = data.num_classes, 
                            heads = 8, 
                            drop_out=drop_out, 
                            learning_rate=LEARNING_RATE, 
                            weighting_method=F1_WEIGHTING_METHOD)
    gat_1_layer = GAT_1_layer(dim_in = data.num_features, 
                            dim_h = embedding_dimension, 
                            dim_out = data.num_classes, 
                            heads = 8, 
                            drop_out=drop_out, 
                            learning_rate=LEARNING_RATE, 
                            weighting_method=F1_WEIGHTING_METHOD)
    return [gat_1_layer, gat_2_layer, gat_3_layer, gat_4_layer]


Train and visualize

In [4]:
from utilities.batch_utilities import BatchUtilities
import numpy as np
batch_utilities = BatchUtilities(data)

for drop_out in [0.6, 0.2]:
    for embedding_dimension in [8, 4, 2]:
        results: dict[str, list[str, float, str, float]] = dict()
        for gat in get_gats(embedding_dimension, drop_out):
            name = gat.get_name()
            result_list = []
            result_list.append('GAT f1')
            result_list.append(gat.fit(data, epochs = NUM_EPOCHS))
            gat.eval()
            out, embedding = gat(data.x, data.edge_index)
            X = embedding.detach().cpu().numpy()
            if VISUALIZE_EMBEDDING: 
                title = name + "-based Embedding"
                batch_utilities.visualize(X, title = title)
            print("Training Classifier")
            batch_utilities.train_classifier(X, max_depth=MAX_DEPTH)
            title = "Confusion matrix for " + name + " Embedding" 
            #batch_utilities.show_confusion_matrix(X,title)
            result_list.append('Classifier f1')
            result_list.append(batch_utilities.get_f1_score(X, weighting_method=F1_WEIGHTING_METHOD))
            results[name] = result_list

        print(f"***************************")
        print(f'* Embedding dimension = {embedding_dimension} *')
        print(f'*      Dropout = {drop_out}      *')
        print(f"***************************")
        for gat_name in results.keys():
            print(f"Results for {gat_name}")
            print(f"\t{results[gat_name][0]} = {np.round(results[gat_name][1],3)}")
            print(f"\t{results[gat_name][2]} = {np.round(results[gat_name][3],3)}")
        print("\n")

Epoch   0 | Train Loss: 2.120 | Train Acc: 16.81% | Test Loss  2.12 | Test Acc: 19.42% | f1  0.152
Epoch 100 | Train Loss: 1.012 | Train Acc: 38.92% | Test Loss  1.05 | Test Acc: 36.42% | f1  0.324
Epoch 200 | Train Loss: 0.949 | Train Acc: 43.07% | Test Loss  0.99 | Test Acc: 41.85% | f1  0.396
Epoch 300 | Train Loss: 0.931 | Train Acc: 45.08% | Test Loss  0.97 | Test Acc: 43.26% | f1  0.411
Epoch 400 | Train Loss: 0.913 | Train Acc: 45.33% | Test Loss  0.97 | Test Acc: 44.37% | f1  0.425
Epoch 500 | Train Loss: 0.899 | Train Acc: 45.84% | Test Loss  0.97 | Test Acc: 43.76% | f1  0.415
Epoch 600 | Train Loss: 0.860 | Train Acc: 49.03% | Test Loss  0.93 | Test Acc: 44.77% | f1  0.433
Epoch 700 | Train Loss: 0.849 | Train Acc: 49.79% | Test Loss  0.89 | Test Acc: 48.39% | f1  0.462
Epoch 800 | Train Loss: 0.816 | Train Acc: 53.66% | Test Loss  0.90 | Test Acc: 51.21% | f1  0.488
Training Classifier
Epoch   0 | Train Loss: 1.344 | Train Acc: 23.04% | Test Loss  1.35 | Test Acc: 23.84% | 